[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week11.ipynb)

# Week 11: LSTMs, GRUs & Sequence Tasks
**Introduction to Deep Learning (HIT)** &middot; Part III: Architectures & Representation Learning

Gated recurrent units; how gates restore gradient flow; LSTM versus GRU; sequence classification and sequence-to-sequence tasks.

**Instructor practice notebook** for the 2-hour practice lesson. Work through the sections below live, running each cell and trying the variations. The student homework is the weekly lab.

### Goals

- Build an LSTM or GRU and compare it to the plain RNN.
- Understand how gates restore gradient flow.
- Apply gated networks to a sequence task.

### Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## 1. RNN vs LSTM vs GRU
Same input; note the parameter counts (gates cost more weights).

In [ ]:
x = torch.randn(8, 20, 1)
for name, layer in [("RNN", nn.RNN(1, 16, batch_first=True)),
                    ("LSTM", nn.LSTM(1, 16, batch_first=True)),
                    ("GRU", nn.GRU(1, 16, batch_first=True))]:
    out = layer(x)[0]
    print(f"{name:4s}: output {tuple(out.shape)}, params {sum(p.numel() for p in layer.parameters())}")

## 2. The cell state and gates
An LSTM returns a hidden state h and a cell state c; weights are 4x hidden (four gates).

In [ ]:
lstm = nn.LSTM(1, 4, batch_first=True)
out, (h, c) = lstm(torch.randn(1, 5, 1))
print("hidden h:", tuple(h.shape), "| cell c:", tuple(c.shape))
print("weight_ih_l0:", tuple(lstm.weight_ih_l0.shape), "= (4*hidden, input): input/forget/cell/output gates")

## 3. Watch the gates open and close
Use LSTMCell to read the forget/input/output gate activations over time.

In [ ]:
torch.manual_seed(0)
cell = nn.LSTMCell(1, 1)
seq = torch.randn(12, 1)
h = torch.zeros(1, 1); c = torch.zeros(1, 1); gates = []
for t in range(12):
    gi = (seq[t:t+1] @ cell.weight_ih.t() + cell.bias_ih + h @ cell.weight_hh.t() + cell.bias_hh).squeeze()
    i, f_, g_, o_ = gi.chunk(4)
    gates.append([torch.sigmoid(f_).item(), torch.sigmoid(i).item(), torch.sigmoid(o_).item()])
    h, c = cell(seq[t:t+1], (h, c))
import numpy as np
gates = np.array(gates)
for k, lab in enumerate(["forget", "input", "output"]):
    plt.plot(gates[:, k], "-o", ms=3, label=lab)
plt.legend(); plt.xlabel("time step"); plt.ylabel("gate value (0-1)"); plt.title("LSTM gate activations"); plt.show()

## 4. Gates preserve gradients over long sequences
Compare the gradient reaching the first step: RNN vs LSTM.

In [ ]:
def first_step_grad(layer, T):
    x = torch.randn(1, T, 1, requires_grad=True)
    out = layer(x)[0]; out[:, -1].sum().backward()
    return x.grad.abs()[:, 0].mean().item()
for T in [10, 50, 100]:
    r = first_step_grad(nn.RNN(1, 8, batch_first=True), T)
    l = first_step_grad(nn.LSTM(1, 8, batch_first=True), T)
    print(f"T={T:3d}:  RNN {r:.2e}   LSTM {l:.2e}")

## 5. Same long-range task, RNN vs LSTM
The label depends on the FIRST step, 40 steps back.

In [ ]:
torch.manual_seed(0)
seqs = torch.randn(500, 40, 1); labels = (seqs[:, 0, 0] > 0).long()   # answer is at step 0
def make(cell):
    class M(nn.Module):
        def __init__(self):
            super().__init__(); self.r = cell(1, 16, batch_first=True); self.head = nn.Linear(16, 2)
        def forward(self, x):
            out, *_ = self.r(x); return self.head(out[:, -1])
    return M()
for name, cell in [("RNN", nn.RNN), ("LSTM", nn.LSTM)]:
    m = make(cell); o = torch.optim.Adam(m.parameters(), 0.01); f = nn.CrossEntropyLoss()
    for _ in range(150):
        o.zero_grad(); f(m(seqs), labels).backward(); o.step()
    print(name, "accuracy:", round((m(seqs).argmax(1) == labels).float().mean().item(), 3))

## 6. A bidirectional RNN
Read the sequence both ways; the output width doubles.

In [ ]:
bi = nn.LSTM(1, 8, batch_first=True, bidirectional=True)
out, _ = bi(torch.randn(4, 10, 1))
print("bidirectional output width:", tuple(out.shape), "(= 2 x hidden)")

## 7. seq2seq, in shapes
An encoder compresses the input; a decoder produces a variable-length output.

In [ ]:
enc = nn.LSTM(1, 16, batch_first=True); dec = nn.LSTM(1, 16, batch_first=True); out_head = nn.Linear(16, 1)
src = torch.randn(2, 7, 1)                  # source length 7
_, (h, c) = enc(src)                        # encode to a state
tgt_in = torch.randn(2, 5, 1)               # target length 5 (teacher forcing input)
dec_out, _ = dec(tgt_in, (h, c))            # decode conditioned on the encoder state
print("encoder state:", tuple(h.shape), "| decoder output:", tuple(out_head(dec_out).shape))

### Mini-exercise
Make the section-5 task depend on the LAST step instead of the first (`labels = (seqs[:, -1, 0] > 0)`). Do the RNN and LSTM both solve it now? Why?

*Try it before revealing the solution below.*

**Solution.**

In [ ]:
torch.manual_seed(0)
seqs2 = torch.randn(500, 40, 1); labels2 = (seqs2[:, -1, 0] > 0).long()    # answer is at the LAST step
for name, cell in [("RNN", nn.RNN), ("LSTM", nn.LSTM)]:
    m = make(cell); o = torch.optim.Adam(m.parameters(), 0.01); f = nn.CrossEntropyLoss()
    for _ in range(150):
        o.zero_grad(); f(m(seqs2), labels2).backward(); o.step()
    print(name, "accuracy:", round((m(seqs2).argmax(1) == labels2).float().mean().item(), 3),
          "(recent info is easy for both)")

**Recap:** gates preserve the gradient signal across long sequences; a GRU is lighter than an LSTM; match the architecture (many-to-one, many-to-many, seq2seq) to the task.

---
Student materials for this week: the lab handout (`labs/week11.html`) and the curated references (`references/week11.html`) in the course site.